In [1]:
import os


In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:
os.chdir('..')

In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
import pandas as pd
from datetime import datetime
import glob
from src.predictor_bot_score.utils.src_util_s3_ import *



In [6]:
@dataclass(frozen=True)
class FeatureEngineeringConfig:
    transformed_data_path : Path
    featured_data_dir     : Path
    featured_raw_data     : Path
    train_data            : Path
    val_data              : Path
    test_data             : Path
    features              : list[str]
    lag_columns           : dict[str, int]
    rolling               : dict[str, int]
    target_column         : str
    bucket_name           : str

In [ ]:
class config_manager:

    def __init__(self,config = CONFIG_PATH):
        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_feature_engineering_config(self) -> FeatureEngineeringConfig:

        config = self.config.feature_engineering

        create_directories([
            config.featured_data_dir,
        ])

        return FeatureEngineeringConfig(
            transformed_data_path = Path(config.transformed_data_path),
            featured_data_dir     = Path(config.featured_data_dir),
            featured_raw_data     = Path(config.featured_raw_data),
            train_data            = Path(config.train_data),
            val_data              = Path(config.validation_data),
            test_data             = Path(config.test_data),
            features              = list(config.features),
            lag_columns           = dict(config.lag_columns),
            rolling               = dict(config.rolling),
            target_column         = config.target_column,
            bucket_name =       self.config.s3_config.bucket_name
        )

In [ ]:
class Feature_engineering:

    def __init__(self, config: FeatureEngineeringConfig):
        self.config = config
        self.BUCKET_NAME = self.config.bucket_name
        self.s3 =  s3_login()
        self.data   = self.read_data()
        self.pipeline_run_id = datetime.now().strftime("%Y_%m_%d_%H")

    def read_data(self):

        try:

            keys = []

            logger.info("=" * 50)
            logger.info("DATA VALIDATION PIPELINE STARTED")
            logger.info("=" * 50)
            
            for page in self.s3.list_objects_v2(Bucket=self.BUCKET_NAME,Prefix='data_validation')['Contents']:
                if page.get('Key').endswith('.csv'):
                    keys.append(page.get('Key'))

            logger.info("S3 . Connection exists ")

            combined_file_key = sorted(keys)[-1]


            obj = self.s3.get_object(Bucket=self.BUCKET_NAME,Key=combined_file_key)
            transformed_val_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

            transformed_val_df["timestamp"] = pd.to_datetime(transformed_val_df["timestamp"], utc=True)

            transformed_val_df["bot_score"] = transformed_val_df["bot_score"].astype(float)
            
            return transformed_val_df
        
        except ClientError as e:
            raise
            
        except Exception as e:
            logger.error(f"Transformation failed: {e}")
            raise


    def build_features(self):
        try:
            logger.info("=" * 50)
            logger.info("FEATURE ENGINEERING PIPELINE STARTED")

            
            logger.info("STEP 1 - LAG FEATURES")
            logger.info("-" * 50)

            for col_name, shift_value in self.config.lag_columns.items():
                self.data[col_name] = self.data["bot_score"].shift(shift_value)
                logger.info(f"Created: {col_name} (shift={shift_value})")
            logger.info("PASSED - Lag features created")

            logger.info("")
            logger.info("STEP 2 - ROLLING STD FEATURES")
            logger.info("-" * 50)

            for col_name, window in self.config.rolling.items():
                self.data[col_name] = self.data["bot_score"].rolling(window).std()
                logger.info(f"Created: {col_name} (window={window})")
            logger.info("PASSED - Rolling features created")

            
            logger.info("STEP 3 - DROP NaN ROWS")
            logger.info("-" * 50)
            before = len(self.data)
            self.data = self.data.dropna(
                subset=self.config.features
            ).reset_index(drop=True)
            after = len(self.data)
            logger.info(f"Dropped {before - after} NaN rows from lag/rolling")
            logger.info(f"PASSED - {after} rows remaining")

            
            logger.info("STEP 4 - SELECT FINAL COLUMNS")
            logger.info("-" * 50)
            final_cols  = self.config.features + [self.config.target_column]
            self.data   = self.data[final_cols]
            logger.info(f"Final columns: {final_cols}")
            logger.info("PASSED - Final columns selected")

            logger.info("=" * 50)
            logger.info("FEATURE ENGINEERING COMPLETE")
            logger.info("=" * 50)

        except Exception as e:
            logger.error(f"Feature engineering failed: {str(e)}")
            raise

        except Exception as e:
            logger.error(f"Failed to save featured data: {str(e)}")
            raise

    def _split_data(self):
        try:
            logger.info("STEP 5 - CHRONOLOGICAL TRAIN/VAL/TEST SPLIT")
            
            # 1. Your existing splitting logic
            total = len(self.data)
            splits = {
                "train": self.data.iloc[:int(total * 0.70)],
                "val":   self.data.iloc[int(total * 0.70):int(total * 0.80)],
                "test":  self.data.iloc[int(total * 0.80):]
            }

            version = datetime.now().strftime("%Y_%m_%d_%H_%M")
            run_folder = f"run__{self.pipeline_run_id}"

            return splits
        except Exception as e:
            raise e
        
    
    def run(self):
        try:
            self.build_features()
            splits    = self._split_data()
            run_folder = f"run__{self.pipeline_run_id}"

            for name, df in splits.items():
                if not isinstance(df, pd.DataFrame):
                    df = pd.DataFrame(df)

                # ── S3 paths — CSV and manifest are separate keys ──
                split_folder = f"feature_engineering/{run_folder}/{name}__{self.pipeline_run_id}"
                csv_key      = f"feature_engineering/{split_folder}/{name}_data.csv"
                manifest_key = f"feature_engineering/{split_folder}/{name}_manifest.json"

                # ── local path — created before writing ────────────
                local_dir = os.path.join(
                    self.config.featured_data_dir,
                    run_folder,
                    name
                )
                os.makedirs(local_dir, exist_ok=True)   # ← create BEFORE writing

                # ── build manifest ─────────────────────────────────
                manifest = save_manifest(
                    pipline_id = self.pipeline_run_id,
                    output_key = csv_key
                )

                # ── save CSV to S3 ─────────────────────────────────
                save_file_s3(
                    df          = df,
                    output_key  = csv_key,
                    BUCKET_NAME = self.BUCKET_NAME,
                    s3_client   = self.s3
                )

                # ── save manifest to S3 (separate key) ────────────
                self_s3_mainfest(
                    manifest    = manifest,
                    output_key  = manifest_key,   # ← different key from CSV
                    BUCKET_NAME = self.BUCKET_NAME,
                    s3_client   = self.s3
                )

                # ── save CSV locally ───────────────────────────────
                df.to_csv(os.path.join(local_dir, f"{name}_data.csv"), index=False)

                # ── save manifest locally ──────────────────────────
                with open(os.path.join(local_dir, "manifest.json"), 'w') as f:
                    json.dump(manifest, f, indent=2)   # ← write directly, no nested subfolders

                logger.info(f"Processed {name}: {len(df)} rows")

            logger.info("=" * 50)
            logger.info("FEATURE ENGINEERING PIPELINE COMPLETED")
            logger.info("=" * 50)

        except Exception as e:
            logger.exception(f"Critical error in Feature Engineering: {e}")
            raise



        

In [44]:
feature_config = config_manager().get_feature_engineering_config()
feature_config = Feature_engineering(feature_config)
feature_config.run()

[2026-07-11 15:44:41,426: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 15:44:41,432: INFO: common: Directory created (or already exists) at: artifacts]
[2026-07-11 15:44:41,432: INFO: common: Directory created (or already exists) at: artifacts/feature_engineering/featured]
[2026-07-11 15:44:41,441: INFO: common: Directory created (or already exists) at: artifacts/feature_engineering/featured/featured_raw_data]
[2026-07-11 15:44:41,443: INFO: common: Directory created (or already exists) at: artifacts/feature_engineering/featured/train_data]
[2026-07-11 15:44:41,443: INFO: common: Directory created (or already exists) at: artifacts/feature_engineering/featured/Validation_data]
[2026-07-11 15:44:41,447: INFO: common: Directory created (or already exists) at: artifacts/feature_engineering/featured/test_data]
[2026-07-11 15:44:41,489: INFO: 4043924616: ==================================================]
[2026-07-11 15:44:41,492: INFO: 4043924616: DATA VALIDA

[2026-07-11 15:44:42,807: INFO: 4043924616: S3 . Connection exists ]
[2026-07-11 15:44:47,644: INFO: 4043924616: ==================================================]
[2026-07-11 15:44:47,644: INFO: 4043924616: FEATURE ENGINEERING PIPELINE STARTED]
[2026-07-11 15:44:47,652: INFO: 4043924616: STEP 1 - LAG FEATURES]
[2026-07-11 15:44:47,652: INFO: 4043924616: --------------------------------------------------]
[2026-07-11 15:44:47,669: INFO: 4043924616: Created: lag_1 (shift=1)]
[2026-07-11 15:44:47,674: INFO: 4043924616: Created: lag_2 (shift=2)]
[2026-07-11 15:44:47,678: INFO: 4043924616: Created: lag_3 (shift=3)]
[2026-07-11 15:44:47,678: INFO: 4043924616: Created: lag_4 (shift=4)]
[2026-07-11 15:44:47,682: INFO: 4043924616: Created: lag_96 (shift=96)]
[2026-07-11 15:44:47,682: INFO: 4043924616: PASSED - Lag features created]
[2026-07-11 15:44:47,687: INFO: 4043924616: ]
[2026-07-11 15:44:47,687: INFO: 4043924616: STEP 2 - ROLLING STD FEATURES]
[2026-07-11 15:44:47,688: INFO: 4043924616

In [59]:
c = yaml_load(CONFIG_PATH)

[2026-06-11 13:58:34,893: INFO: common: yaml file: config\config.yaml loaded successfully]


In [61]:
c.feature_engineering.test_data

'artifacts/feature_engineering/featured/test_data'